In [1]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [2]:
# !hf auth login

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [4]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

* Figure out which layers to replace
* write function like LoRA linear for attenton and mha modules
* write function to replce this model layers to above written lora modules while keeping original weights frozen
* make this all into a moddule for the math

### LoRA from Scratch

In [5]:
import math
import torch
from torch import nn

In [6]:
layer = nn.Linear(in_features=3, out_features=2, bias=True)
input_tensor = torch.tensor([1., 2., 3.])
input_tensor

tensor([1., 2., 3.])

In [7]:
with torch.no_grad():
  layer.weight = nn.Parameter(torch.tensor([[0.1, 0.2, 0.3],
                                            [0.4, 0.5, 0.6]]))
  layer.bias = nn.Parameter(torch.tensor([0.7, 0.8]))

In [8]:
class LoRALinear(nn.Module):
  def __init__(self, base:nn.Linear, r:int, alpha:float = 16.0):
    super().__init__()
    self.r = r
    self.alpha = alpha
    self.base = base
    self.scaling = alpha / r

    # original W (frozen)
    self.base.requires_grad_(False)
    if self.base.bias is not None:
      self.base.bias.requires_grad_(False)

    self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
    self.lora_B = nn.Parameter(torch.empty(base.out_features, r))

    nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
    nn.init.zeros_(self.lora_B)


  def foward(self, x):
    base_output = self.base(x)    # Wx
    lora_update = F.linear(F.linear(x, self.lora_A), self.lora_B)

    return base_output + (lora_update * self.scaling)

### DoRA from scrath - same as LoRA but weight decomposition into magnitude and direction components

Dora wight scaling -> ``` W_dora = (m / ||W + ΔW||) * (W + ΔW) ```

In [9]:
class DoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float = 16.0):
        super().__init__()
        self.r = r
        self.alpha = alpha
        self.base = base
        self.scaling = alpha / r

        # Freeze original weights and bias
        self.base.requires_grad_(False)
        if self.base.bias is not None:
            self.base.bias.requires_grad_(False)

        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.empty(base.out_features, r))

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

        W = base.weight.detach()
        self.magnitude = nn.Parameter(W.norm(p=2, dim=1, keepdim=True))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Adapted weight = W + LoRA update
        W = self.base.weight
        lora_update = (self.lora_B @ self.lora_A) * self.scaling

        W_adapted = W + lora_update
        W_norm = W_adapted.norm(p=2, dim=1, keepdim=True)
        W_dora = (self.magnitude / W_norm) * W_adapted

        return F.linear(x, W_dora, self.base.bias)

#### Making the class to use DoRA instead of LLama GQA

In [15]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional
from transformers.models.llama.modeling_llama import LlamaAttention, apply_rotary_pos_emb
from transformers import LlamaForCausalLM

In [23]:
from transformers.models.llama.modeling_llama import LlamaAttention
from typing import Optional

class DoRALlamaMHA(nn.Module):

    def __init__(self, original_attn, rotary_emb, r: int, alpha: float = 16.0):
        super().__init__()

        cfg = original_attn.config
        self.hidden_size   = cfg.hidden_size
        self.num_heads     = cfg.num_attention_heads
        self.num_kv_heads  = cfg.num_key_value_heads
        self.head_dim      = cfg.hidden_size // cfg.num_attention_heads
        self.num_kv_groups = self.num_heads // self.num_kv_heads

        self.q_proj = DoRALinear(original_attn.q_proj, r, alpha)
        self.k_proj = DoRALinear(original_attn.k_proj, r, alpha)
        self.v_proj = DoRALinear(original_attn.v_proj, r, alpha)
        self.o_proj = DoRALinear(original_attn.o_proj, r, alpha)

        self.rotary_emb = rotary_emb

    @staticmethod
    def _repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
        if n_rep == 1:
            return x
        B, num_kv_heads, S, head_dim = x.shape
        return (
            x[:, :, None, :, :]
            .expand(B, num_kv_heads, n_rep, S, head_dim)
            .reshape(B, num_kv_heads * n_rep, S, head_dim)
        )

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_value=None,
        output_attentions: bool = False,
        use_cache: bool = False,
        cache_position: Optional[torch.LongTensor] = None,
        **kwargs,
    ):
        B, S, _ = hidden_states.shape

        Q = self.q_proj(hidden_states)
        K = self.k_proj(hidden_states)
        V = self.v_proj(hidden_states)

        Q = Q.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)

        cos, sin = self.rotary_emb(V, position_ids)
        Q, K = apply_rotary_pos_emb(Q, K, cos, sin)

        if past_key_value is not None:
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}
            K, V = past_key_value.update(K, V, self.layer_idx, cache_kwargs)

        K = self._repeat_kv(K, self.num_kv_groups)
        V = self._repeat_kv(V, self.num_kv_groups)

        attn_output = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attention_mask,
            dropout_p=0.0,
            is_causal=(attention_mask is None),
        )

        attn_output = attn_output.transpose(1, 2).contiguous().view(B, S, -1)
        attn_output = self.o_proj(attn_output)

        return attn_output, None, past_key_value

#### Replace llama gqa with dora dqa and see changes

In [24]:
def apply_dora_to_llama(model, r: int = 16, alpha: float = 32.0):
    for layer_idx, layer in enumerate(model.model.layers):
        rotary_emb = getattr(layer.self_attn, "rotary_emb", None) or getattr(layer, "rotary_emb", None)
        dora_attn = DoRALlamaMHA(layer.self_attn, rotary_emb, r=r, alpha=alpha)
        dora_attn.layer_idx = layer_idx
        layer.self_attn = dora_attn
    return model

model = LlamaForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B",
    torch_dtype=torch.float16,
    device_map="auto",
)
model = apply_dora_to_llama(model, r=16, alpha=32.0)
model

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): DoRALlamaMHA(
          (q_proj): DoRALinear(
            (base): Linear(in_features=4096, out_features=4096, bias=False)
          )
          (k_proj): DoRALinear(
            (base): Linear(in_features=4096, out_features=1024, bias=False)
          )
          (v_proj): DoRALinear(
            (base): Linear(in_features=4096, out_features=1024, bias=False)
          )
          (o_proj): DoRALinear(
            (base): Linear(in_features=4096, out_features=4096, bias=False)
          )
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_l

### Modularizing for convinence

In [22]:
%%writefile dora.py

import math
from typing import Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import LlamaForCausalLM
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb


class DoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float = 16.0):
        super().__init__()
        self.r = r
        self.alpha = alpha
        self.base = base
        self.scaling = alpha / r

        self.base.requires_grad_(False)
        if self.base.bias is not None:
            self.base.bias.requires_grad_(False)

        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.empty(base.out_features, r))

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

        W = base.weight.detach()
        self.magnitude = nn.Parameter(W.norm(p=2, dim=1, keepdim=True))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        W = self.base.weight
        lora_update = (self.lora_B @ self.lora_A) * self.scaling
        W_adapted = W + lora_update

        W_norm = W_adapted.norm(p=2, dim=1, keepdim=True)
        W_dora = (self.magnitude / W_norm) * W_adapted

        return F.linear(x, W_dora, self.base.bias)


class DoRALlamaMHA(nn.Module):
    def __init__(self, original_attn, rotary_emb, r: int, alpha: float = 16.0):
        super().__init__()

        cfg = original_attn.config
        self.hidden_size = cfg.hidden_size
        self.num_heads = cfg.num_attention_heads
        self.num_kv_heads = cfg.num_key_value_heads
        self.head_dim = cfg.hidden_size // cfg.num_attention_heads
        self.num_kv_groups = self.num_heads // self.num_kv_heads

        self.q_proj = DoRALinear(original_attn.q_proj, r, alpha)
        self.k_proj = DoRALinear(original_attn.k_proj, r, alpha)
        self.v_proj = DoRALinear(original_attn.v_proj, r, alpha)
        self.o_proj = DoRALinear(original_attn.o_proj, r, alpha)

        self.rotary_emb = rotary_emb
        self.layer_idx: Optional[int] = getattr(original_attn, "layer_idx", None)

    @staticmethod
    def _repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
        if n_rep == 1:
            return x
        B, num_kv_heads, S, head_dim = x.shape
        return (
            x[:, :, None, :, :]
            .expand(B, num_kv_heads, n_rep, S, head_dim)
            .reshape(B, num_kv_heads * n_rep, S, head_dim)
        )

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_value=None,
        output_attentions: bool = False,
        use_cache: bool = False,
        cache_position: Optional[torch.LongTensor] = None,
        **kwargs,
    ):
        B, S, _ = hidden_states.shape

        Q = self.q_proj(hidden_states)
        K = self.k_proj(hidden_states)
        V = self.v_proj(hidden_states)

        Q = Q.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)

        cos, sin = self.rotary_emb(V, position_ids)
        Q, K = apply_rotary_pos_emb(Q, K, cos, sin)

        if past_key_value is not None:
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}
            K, V = past_key_value.update(K, V, self.layer_idx, cache_kwargs)

        K = self._repeat_kv(K, self.num_kv_groups)
        V = self._repeat_kv(V, self.num_kv_groups)

        attn_output = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attention_mask,
            dropout_p=0.0,
            is_causal=(attention_mask is None),
        )

        attn_output = attn_output.transpose(1, 2).contiguous().view(B, S, -1)
        attn_output = self.o_proj(attn_output)

        return attn_output, None, past_key_value


def apply_dora_to_llama(model: LlamaForCausalLM, r: int = 16, alpha: float = 32.0):
    for layer_idx, layer in enumerate(model.model.layers):
        rotary_emb = getattr(layer.self_attn, "rotary_emb", None) or getattr(layer, "rotary_emb", None)
        dora_attn = DoRALlamaMHA(layer.self_attn, rotary_emb, r=r, alpha=alpha)
        dora_attn.layer_idx = layer_idx
        layer.self_attn = dora_attn
    return model


def dora_trainable_params(model) -> List[nn.Parameter]:
    return [p for n, p in model.named_parameters() if p.requires_grad]


def dora_param_count(model) -> dict:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "total": total,
        "trainable": trainable,
        "frozen": total - trainable,
        "trainable_pct": round(100 * trainable / total, 4),
    }

Writing dora.py


In [26]:
%%writefile test_dora.py

import sys
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, "/content")
from dora import DoRALinear, DoRALlamaMHA, apply_dora_to_llama, dora_param_count

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"Running on {DEVICE} ({DTYPE})\n")

PASS = "✅ PASS"
FAIL = "❌ FAIL"


def test_trainable_params():
    base  = nn.Linear(64, 128, bias=True)
    layer = DoRALinear(base, r=4, alpha=8.0)

    frozen    = [n for n, p in layer.named_parameters() if not p.requires_grad]
    trainable = [n for n, p in layer.named_parameters() if p.requires_grad]

    ok = (set(frozen) == {"base.weight", "base.bias"}) and (set(trainable) == {"lora_A", "lora_B", "magnitude"})
    print(f"TEST 1 — Trainable/frozen split       {PASS if ok else FAIL}")
    if not ok:
        print(f"  frozen={frozen}  trainable={trainable}")

test_trainable_params()


def test_identity_at_init():
    torch.manual_seed(0)
    base     = nn.Linear(64, 128, bias=False)
    layer    = DoRALinear(base, r=4, alpha=8.0).to(DTYPE)
    x        = torch.randn(2, 10, 64, dtype=DTYPE)
    out_dora = layer(x)
    out_base = F.linear(x, base.weight.to(DTYPE))
    ok       = torch.allclose(out_dora, out_base, atol=1e-4)
    print(f"TEST 2 — Identity at init             {PASS if ok else FAIL}")
    if not ok:
        print(f"  max_diff={(out_dora - out_base).abs().max().item():.6f}")

test_identity_at_init()


def test_magnitude_init():
    base          = nn.Linear(64, 128, bias=False)
    layer         = DoRALinear(base, r=4, alpha=8.0)
    expected_vals = base.weight.detach().norm(p=2, dim=1, keepdim=True)
    ok = (layer.magnitude.shape == (128, 1)) and torch.allclose(layer.magnitude.data, expected_vals, atol=1e-6)
    print(f"TEST 3 — Magnitude init shape/value   {PASS if ok else FAIL}")

test_magnitude_init()


def test_output_shape():
    B, S, IN, OUT = 2, 7, 64, 128
    base  = nn.Linear(IN, OUT)
    layer = DoRALinear(base, r=8).to(DTYPE)
    x     = torch.randn(B, S, IN, dtype=DTYPE)
    out   = layer(x)
    ok    = (out.shape == (B, S, OUT))
    print(f"TEST 4 — Output shape {tuple(out.shape):<20} {PASS if ok else FAIL}")

test_output_shape()


def test_gradients():
    base  = nn.Linear(32, 64)
    layer = DoRALinear(base, r=4).to(torch.float32)
    x     = torch.randn(2, 5, 32)
    layer(x).sum().backward()
    ok = (layer.lora_A.grad is not None and
          layer.lora_B.grad is not None and
          layer.magnitude.grad is not None and
          base.weight.grad is None)
    print(f"TEST 5 — Gradients (train/frozen)     {PASS if ok else FAIL}")

test_gradients()


def test_unit_direction():
    torch.manual_seed(42)
    base  = nn.Linear(64, 128, bias=False)
    layer = DoRALinear(base, r=4, alpha=8.0).to(torch.float32)
    with torch.no_grad():
        layer.lora_B.normal_(0, 0.01)
    W         = layer.base.weight
    W_adapted = W + (layer.lora_B @ layer.lora_A) * layer.scaling
    dir_norms = (W_adapted / W_adapted.norm(p=2, dim=1, keepdim=True)).norm(p=2, dim=1)
    ok        = torch.allclose(dir_norms, torch.ones_like(dir_norms), atol=1e-5)
    print(f"TEST 6 — Unit-direction per neuron    {PASS if ok else FAIL}")

test_unit_direction()


def test_llama_swap():
    try:
        from transformers import LlamaConfig, LlamaForCausalLM
        cfg = LlamaConfig(
            hidden_size=256,
            intermediate_size=512,
            num_hidden_layers=2,
            num_attention_heads=8,
            num_key_value_heads=2,
            max_position_embeddings=64,
            vocab_size=1000,
        )
        model  = LlamaForCausalLM(cfg)
        model  = apply_dora_to_llama(model, r=4, alpha=8.0)
        ok     = all(isinstance(layer.self_attn, DoRALlamaMHA) for layer in model.model.layers)
        counts = dora_param_count(model)
        print(f"TEST 7 — Llama attn layer swap        {PASS if ok else FAIL}")
        print(f"         trainable {counts['trainable']:,} / {counts['total']:,} params  ({counts['trainable_pct']}%)")
    except ImportError:
        print("TEST 7 — SKIPPED (transformers not installed)")

test_llama_swap()


def test_llama_forward():
    try:
        from transformers import LlamaConfig, LlamaForCausalLM
        cfg = LlamaConfig(
            hidden_size=256,
            intermediate_size=512,
            num_hidden_layers=2,
            num_attention_heads=8,
            num_key_value_heads=2,
            max_position_embeddings=64,
            vocab_size=1000,
        )
        model     = LlamaForCausalLM(cfg)
        model     = apply_dora_to_llama(model, r=4, alpha=8.0)
        model.eval()
        input_ids = torch.randint(0, 1000, (1, 16))
        with torch.no_grad():
            out = model(input_ids)
        ok = out.logits.shape == (1, 16, 1000)
        print(f"TEST 8 — Llama forward pass shape     {PASS if ok else FAIL}  {tuple(out.logits.shape)}")
    except ImportError:
        print("TEST 8 — SKIPPED (transformers not installed)")

test_llama_forward()

print("\nAll tests complete.")

Writing test_dora.py
